# Whisper-LLaDA 구동 확인 (진행/중단 판정의 핵심)

참고: CLAUDE.md 3.3절, `docs/setup_drax_whisper_llada.md` 3·4절 (할 일 4)

확인할 것: ① 체크포인트 로드 시 빠진 키가 없는지 ② 순수 확산·semi-AR 정상 전사 ③ GPU 메모리·클립당 시간

**별도 Colab 세션, A100 40GB에서 실행한다.** 처음에 약 19GB(추정)를 받는다.

In [ ]:
# 공통 설정: GitHub에서 코드 받기 + Drive 연결
# Drive에는 데이터·모델 캐시·결과를 둔다 (세션이 끊겨도 남음)
import os, sys
from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/diffusion-asr-hallucination'
if os.path.exists(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/alsgur4082/diffusion-asr-hallucination.git {REPO}
sys.path.insert(0, REPO)

DRIVE = '/content/drive/MyDrive/dah'   # 데이터·캐시·결과 루트
DATA = f'{DRIVE}/data'
RESULTS = f'{DRIVE}/results'
os.environ['HF_HOME'] = f'{DRIVE}/hf_cache'  # 모델 import 전에 설정해야 적용됨
os.makedirs(RESULTS, exist_ok=True)

!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# conda 대신 pip. 추론에는 k2가 필요 없다 (설치 가이드 3절)
!pip install -q torch==2.7.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu126
!pip install -q transformers==4.53.2 peft==0.3.0 soundfile pyyaml tqdm jiwer scipy gdown
!git clone -q https://github.com/liuzhan22/Diffusion-ASR.git /content/Diffusion-ASR 2>/dev/null || true
# torch를 새로 설치했으면: 런타임 → 세션 다시 시작 후 첫 셀부터 다시 실행

In [ ]:
# 학습 체크포인트 (Google Drive 공개 파일) → 내 Drive에 한 번만 받기
CKPT = f'{DRIVE}/ckpt/whisper-large-v3-llada-checkpoint_best_epoch_143.pth'
os.makedirs(os.path.dirname(CKPT), exist_ok=True)
if not os.path.exists(CKPT):
    !gdown 1btAsCXTHQApnRcl_k5Wfn0zSD08yXHZh -O {CKPT}
!ls -lh {CKPT}

In [ ]:
# 스모크 테스트용 소량 데이터 (LibriSpeech test-clean 5개)
from src.data.download import librispeech
ls_paths, ls_refs = librispeech(DATA, 'test-clean', n=5)
print(len(ls_paths), ls_paths[0], ls_refs[0])

## 1. 모델 로드 + 체크포인트 키 확인

`inference.py`와 같은 순서로 로드하되, `strict=False`가 숨기는 누락 키를 직접 출력한다.

In [ ]:
import time, torch
os.chdir('/content/Diffusion-ASR'); sys.path.insert(0, '/content/Diffusion-ASR')
from transformers import WhisperFeatureExtractor
from models.WhisperLLaDA import WhisperLLaDA

torch.cuda.reset_peak_memory_stats()
model = WhisperLLaDA(whisper_model='openai/whisper-large-v3', llada_model='GSAI-ML/LLaDA-8B-Instruct',
                     gen_len=128, lora=True, lora_rank=8, lora_alpha=32, lora_dropout=0.1).to('cuda').eval()
fe = WhisperFeatureExtractor.from_pretrained('openai/whisper-large-v3')

ckpt = torch.load(CKPT, map_location='cuda', weights_only=False)
missing, unexpected = model.load_state_dict(ckpt['model'], strict=False)
print('체크포인트 키 개수:', len(ckpt['model']))
print('unexpected (비어 있어야 함):', unexpected[:10], len(unexpected))
bad = [k for k in missing if any(s in k for s in ('lora', 'adapter', 'linear_proj', 'speech_query'))]
print('학습된 부분 중 빠진 키 (비어 있어야 함):', bad[:10], len(bad))
print(f'로드 후 GPU 메모리: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB')

## 2. 순수 확산 vs semi-AR

In [ ]:
from src.data.audio import load_mono_16k
from src.runners.base import DecodeConfig, TranscriptionResult, save_results
from src.metrics.basic import wer

def run(paths, block_length, steps=128, gen_len=128):
    decode_cfg = {'mode': 'decoding', 'steps': steps, 'block_length': block_length}
    cfg = DecodeConfig(model='whisper-llada', language='en', temperature=0.0, num_steps=steps, seed=0,
                       selection='low_confidence', extra={'gen_len': gen_len, **decode_cfg})
    torch.manual_seed(cfg.seed)
    out = []
    for p in paths:
        audio = load_mono_16k(p)
        log_mel = fe(audio, sampling_rate=16000, return_tensors='pt')['input_features'].to('cuda')
        t0 = time.time()
        pred = model.generate(samples={'spectrogram': log_mel}, decode_cfg=decode_cfg)[0]
        torch.cuda.synchronize()
        out.append(TranscriptionResult(audio_path=p, transcript=pred, elapsed_sec=time.time() - t0, config=cfg))
    return out

summary = {}
for name, bl in [('pure_diffusion', 128), ('semi_ar_64', 64), ('semi_ar_32', 32)]:
    torch.cuda.reset_peak_memory_stats()
    res = run(ls_paths, bl)
    save_results(res, f'{RESULTS}/whisper-llada_smoke_{name}.jsonl')
    summary[name] = {**wer(ls_refs, [r.transcript for r in res]),
                     'sec_per_clip': sum(r.elapsed_sec for r in res) / len(res),
                     'peak_gb': torch.cuda.max_memory_allocated() / 1e9}
    print(f'== {name}')
    for r, ref in zip(res, ls_refs):
        print('REF:', ref); print('HYP:', r.transcript)
    print(summary[name]); print()

## 3. 비음성 1~2클립

정상 음성 결과가 괜찮으면 비음성 입력에서 무엇을 내는지도 본다 (00 노트북에서 UrbanSound를 이미 받았다면 재사용됨).

In [ ]:
from src.data.download import urbansound8k
us_paths, _, us_classes = urbansound8k(DATA, per_class=1)
for name, bl in [('pure_diffusion', 128), ('semi_ar_32', 32)]:
    res = run(us_paths[:3], bl)
    for r, c in zip(res, us_classes):
        print(f'{name:15s} {c:20s} -> {r.transcript!r}')